# LTL Safety Colour Bomb

This notebook focuses on MASA's `ltl_safety` constraint. We will use Colour Bomb Grid World labels as a trace, advance a DFA over those labels, and inspect exactly when violations appear in `info`.

The matching static docs page is at `docs/Tutorials/LTL-Safety/Colour Bomb.md`.

## CPU-first setup

Keep this notebook portable and quiet before importing MASA/JAX modules.

In [ ]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

from pprint import pprint

ACTION_NAMES = {0: "left", 1: "right", 2: "down", 3: "up", 4: "stay"}

## Imports

The tutorial uses the small 9x9 `colour_bomb_grid_world` for the main examples, then briefly switches to `colour_bomb_grid_world_v2` for the larger medic-recovery DFA.

In [ ]:
from IPython.display import SVG, display

from masa.plugins.helpers import load_plugins
from masa.common.ltl import Atom, DFA
from masa.common.utils import make_env
from masa.envs.tabular.colour_bomb_grid_world import (
    BOMB_STATES,
    BLUE_STATES,
    GREEN_STATES,
    GRID_SIZE,
    PINK_STATES,
    START_STATE,
    WALL_STATES,
    YELLOW_STATES,
    label_fn,
)
from masa.envs.tabular.colour_bomb_grid_world_v2 import (
    BOMB_STATES as BOMB_STATES_V2,
    BLUE_STATES as BLUE_STATES_V2,
    GREEN_STATES as GREEN_STATES_V2,
    GRID_SIZE as GRID_SIZE_V2,
    MEDIC_STATES as MEDIC_STATES_V2,
    PINK_STATES as PINK_STATES_V2,
    RED_STATES as RED_STATES_V2,
    START_STATES as START_STATES_V2,
    WALL_STATES as WALL_STATES_V2,
    YELLOW_STATES as YELLOW_STATES_V2,
    label_fn as label_fn_v2,
)
from masa.examples.colour_bomb_grid_world.property_2 import make_dfa as make_diffusion_dfa
from masa.examples.colour_bomb_grid_world.property_3 import make_dfa as make_medic_recovery_dfa
from notebooks.tutorials.helpers.ltl_safety_colour_bomb import (
    render_colour_bomb_trace_svg,
    render_dfa_diagram_svg,
    render_ltl_rollout_timeline_svg,
)

load_plugins()

## Define and Load DFA Properties

A MASA LTL-safety property is represented by a DFA. Accepting states are unsafe states: when the monitor enters one, `info["constraint"]["step"]["violation"]` becomes `1.0` and the episode-level `satisfied` metric becomes `0.0`.

The first DFA is defined inline. The second is loaded from the existing Colour Bomb examples.

In [ ]:
def make_never_bomb_dfa():
    dfa = DFA([0, 1], 0, [1])
    dfa.add_edge(0, 1, Atom("bomb"))
    return dfa

never_bomb_dfa = make_never_bomb_dfa()
diffusion_dfa = make_diffusion_dfa()

print("never_bomb states:", never_bomb_dfa.states, "accepting:", never_bomb_dfa.accepting)
print("property_2 states:", diffusion_dfa.states, "accepting:", diffusion_dfa.accepting)

In [ ]:
display(SVG(render_dfa_diagram_svg(
    title="Inline DFA: never observe bomb",
    states=[0, 1],
    accepting=[1],
    edges=[(0, 1, "bomb"), (1, 1, "any label")],
    state_labels={0: "safe", 1: "unsafe"},
)))

display(SVG(render_dfa_diagram_svg(
    title="Loaded property_2 DFA: stay on bomb once before leaving",
    states=[0, 1, 2, 3],
    accepting=[3],
    edges=[(0, 1, "bomb"), (1, 2, "bomb"), (1, 3, "not bomb"), (2, 0, "not bomb")],
    state_labels={0: "safe", 1: "armed", 2: "diffused", 3: "unsafe"},
)))

## Build an `ltl_safety` Environment

`obs_type="dict"` keeps the original tabular state and automaton state visible as separate fields. This is ideal for tutorials because `obs["orig"]` is the Colour Bomb state and `obs["automaton"]` is the DFA state index.

In [ ]:
def build_ltl_env(env_id, labels, dfa, max_episode_steps=30):
    return make_env(
        env_id,
        "ltl_safety",
        max_episode_steps,
        label_fn=labels,
        dfa=dfa,
        obs_type="dict",
    )


def row_from(event, obs, info, action=None, reward=None, terminated=False, truncated=False):
    constraint = info.get("constraint", {})
    return {
        "event": event,
        "action": action,
        "action_name": ACTION_NAMES.get(action) if action is not None else None,
        "obs_orig": int(obs["orig"]),
        "obs_automaton": int(obs["automaton"]),
        "automaton_state": int(info.get("automaton_state", obs["automaton"])),
        "labels": sorted(info.get("labels", [])),
        "reward": None if reward is None else float(reward),
        "terminated": bool(terminated),
        "truncated": bool(truncated),
        "constraint_step": dict(constraint.get("step", {})),
        "constraint_episode": dict(constraint.get("episode", {})),
    }


def run_ltl_script(env_id, labels, dfa, *, seed, actions, max_episode_steps=30):
    env = build_ltl_env(env_id, labels, dfa, max_episode_steps=max_episode_steps)
    obs, info = env.reset(seed=seed)
    rows = [row_from("reset", obs, info)]

    for step, action in enumerate(actions, start=1):
        obs, reward, terminated, truncated, info = env.step(action)
        rows.append(
            row_from(
                f"step_{step}",
                obs,
                info,
                action=action,
                reward=reward,
                terminated=terminated,
                truncated=truncated,
            )
        )
        if terminated or truncated:
            break

    env.close()
    return rows

## Inline DFA: Never Hit a Bomb

With seed `1`, four right actions move from the start state to state `78`, which has label `bomb`. The never-bomb DFA moves from safe state `0` to accepting unsafe state `1` on that label.

In [ ]:
never_bomb_rows = run_ltl_script(
    "colour_bomb_grid_world",
    label_fn,
    make_never_bomb_dfa(),
    seed=1,
    actions=[1, 1, 1, 1],
)

pprint(never_bomb_rows)

assert never_bomb_rows[-1]["labels"] == ["bomb"]
assert never_bomb_rows[-1]["automaton_state"] == 1
assert never_bomb_rows[-1]["constraint_step"]["violation"] == 1.0
assert never_bomb_rows[-1]["constraint_episode"]["cum_unsafe"] == 1.0
assert never_bomb_rows[-1]["constraint_episode"]["satisfied"] == 0.0

In [ ]:
display(SVG(render_colour_bomb_trace_svg(
    never_bomb_rows,
    title="Seed 1 reaches a bomb in four right moves",
    grid_size=GRID_SIZE,
    start_states=[START_STATE],
    wall_states=WALL_STATES,
    bomb_states=BOMB_STATES,
    goal_states=GREEN_STATES + YELLOW_STATES + BLUE_STATES + PINK_STATES,
)))

display(SVG(render_ltl_rollout_timeline_svg(
    never_bomb_rows,
    title="Never-bomb monitor: violation on the bomb label",
)))

## Loaded DFA: Bomb Diffusion

`property_2` is more nuanced. Hitting a bomb is not immediately unsafe. The unsafe event is hitting a bomb and then leaving it immediately. Staying on the bomb for one extra step diffuses it.

In [ ]:
diffusion_violation_rows = run_ltl_script(
    "colour_bomb_grid_world",
    label_fn,
    make_diffusion_dfa(),
    seed=1,
    actions=[1, 1, 1, 1, 0],
)

diffusion_safe_rows = run_ltl_script(
    "colour_bomb_grid_world",
    label_fn,
    make_diffusion_dfa(),
    seed=1,
    actions=[1, 1, 1, 1, 4, 0],
)

print("Immediate leave:")
pprint(diffusion_violation_rows)
print("\nStay once, then leave:")
pprint(diffusion_safe_rows)

assert diffusion_violation_rows[-1]["automaton_state"] == 3
assert diffusion_violation_rows[-1]["constraint_step"]["violation"] == 1.0
assert diffusion_violation_rows[-1]["constraint_episode"]["satisfied"] == 0.0

assert diffusion_safe_rows[-1]["automaton_state"] == 0
assert diffusion_safe_rows[-1]["constraint_step"]["violation"] == 0.0
assert diffusion_safe_rows[-1]["constraint_episode"]["cum_unsafe"] == 0.0
assert diffusion_safe_rows[-1]["constraint_episode"]["satisfied"] == 1.0

In [ ]:
display(SVG(render_ltl_rollout_timeline_svg(
    diffusion_violation_rows,
    title="property_2 monitor: leaving bomb immediately violates",
)))

display(SVG(render_ltl_rollout_timeline_svg(
    diffusion_safe_rows,
    title="property_2 monitor: one extra bomb step avoids violation",
)))

## V2 Extension: Medic Recovery

`colour_bomb_grid_world_v2` adds medic labels and uses a larger loaded DFA, `property_3`. After a bomb, the monitor expects recovery by reaching and remaining on `medic`; if that does not happen in time, it reaches accepting violation state `21`.

The script below starts at state `16` with seed `14`, reaches a bomb, and waits without reaching a medic.

In [ ]:
medic_recovery_rows = run_ltl_script(
    "colour_bomb_grid_world_v2",
    label_fn_v2,
    make_medic_recovery_dfa(),
    seed=14,
    actions=[2, 2, 2, 2] + [4] * 11,
    max_episode_steps=60,
)

pprint(medic_recovery_rows)

assert any(row["automaton_state"] == 21 for row in medic_recovery_rows)
assert medic_recovery_rows[-1]["constraint_step"]["violation"] == 1.0
assert medic_recovery_rows[-1]["constraint_episode"]["cum_unsafe"] >= 1.0
assert medic_recovery_rows[-1]["constraint_episode"]["satisfied"] == 0.0

In [ ]:
display(SVG(render_colour_bomb_trace_svg(
    medic_recovery_rows,
    title="V2 seed 14 reaches a bomb, then waits without medic",
    grid_size=GRID_SIZE_V2,
    start_states=START_STATES_V2,
    wall_states=WALL_STATES_V2,
    bomb_states=BOMB_STATES_V2,
    goal_states=GREEN_STATES_V2 + YELLOW_STATES_V2 + RED_STATES_V2 + BLUE_STATES_V2 + PINK_STATES_V2,
    medic_states=MEDIC_STATES_V2,
)))

display(SVG(render_ltl_rollout_timeline_svg(
    medic_recovery_rows,
    title="property_3 monitor: accepting state 21 after missed recovery",
)))

## What to Take Away

- `ltl_safety` treats labels as a sequence, not just isolated per-step costs.
- The DFA state is part of the observation when `obs_type="dict"`, so agents can learn over the product of environment state and automaton state.
- `info["automaton_state"]` mirrors the monitor state, while `info["constraint"]["step"]` and `info["constraint"]["episode"]` report violation and satisfaction metrics.
- Loading a DFA from a module uses the same runtime path as defining one inline; only the property changes.